# Paired bootstrap and per-class analysis
Uses your saved predictions; no retraining. Upload the original results backup. Intervals condition on fitted models and are not adjusted for multiple comparisons. Five-seed results share the same resampled test articles; seeds are not treated as independent test observations.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()  # Choose your original Four_Language_Results_Backup.zip
archive = next(iter(uploaded))
root = Path('/content/reanalysis')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        parts = Path(name).parts
        if 'languages' in parts and name.endswith('/results/test_predictions.csv'):
            rel = Path(*parts[parts.index('languages'):])
            if '..' in rel.parts: raise ValueError('Unsafe ZIP path')
            target = root / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(z.read(name))
assert len(list(root.glob('languages/*/results/test_predictions.csv'))) == 4


In [ ]:
import os, subprocess, sys
(root/'analyse.py').write_text("from pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import f1_score,classification_report\nROOT=Path(__file__).resolve().parent\nLANGS={'yor':'Yorùbá','ibo':'Igbo','hau':'Hausa','pcm':'Nigerian Pidgin'}\nrows=[]; classes=[]; disagreements=[]\nfor li,(code,name) in enumerate(LANGS.items()):\n d=pd.read_csv(ROOT/'languages'/code/'results/test_predictions.csv')\n base=d[(d.fraction==1)&(d.approach=='Word')].sort_values('row_id')\n truth=base.true.to_numpy(); ids=base.row_id.to_numpy(); labels=sorted(set(truth)); n=len(truth)\n rng=np.random.default_rng(20261019+li)\n weights=rng.multinomial(n,np.full(n,1/n),size=10000).astype(float)\n def scores(pred,w):\n  vals=[]\n  for label in labels:\n   t=(truth==label);p=(pred==label)\n   tp=w@(t&p);den=w@t+w@p\n   vals.append(np.divide(2*tp,den,out=np.zeros_like(tp),where=den>0))\n  return np.mean(vals,axis=0)*100\n for fraction in [.25,.5,1.]:\n  boot={};point={};preds={}\n  for approach in ['Word','Character']:\n   arrays=[];points=[]\n   for seed,g in d[(d.fraction==fraction)&(d.approach==approach)].groupby('seed'):\n    g=g.sort_values('row_id');assert np.array_equal(g.row_id,ids) and np.array_equal(g.true,truth)\n    pred=g.predicted.to_numpy();arrays.append(scores(pred,weights));points.append(f1_score(truth,pred,average='macro',labels=labels)*100)\n    assert np.isclose(scores(pred,np.ones((1,n)))[0],points[-1])\n    if fraction==1:\n     preds[approach]=pred\n     report=classification_report(truth,pred,labels=labels,output_dict=True,zero_division=0)\n     for label in labels:classes.append(dict(code=code,language=name,approach=approach,category=label,**report[label]))\n   boot[approach]=np.mean(arrays,axis=0);point[approach]=np.mean(points)\n  delta=boot['Character']-boot['Word'];lo,hi=np.quantile(delta,[.025,.975])\n  rows.append(dict(code=code,language=name,fraction=fraction,n_test=n,word=point['Word'],character=point['Character'],difference=point['Character']-point['Word'],ci_low=lo,ci_high=hi,bootstrap_samples=10000))\n  if fraction==1:\n   for i in np.flatnonzero(preds['Word']!=preds['Character']):disagreements.append(dict(code=code,row_id=ids[i],true=truth[i],word=preds['Word'][i],character=preds['Character'][i],word_correct=preds['Word'][i]==truth[i],character_correct=preds['Character'][i]==truth[i]))\nout=pd.DataFrame(rows);out.to_csv(ROOT/'bootstrap_intervals.csv',index=False)\npd.DataFrame(classes).to_csv(ROOT/'per_class_full_data.csv',index=False)\npd.DataFrame(disagreements).to_csv(ROOT/'full_data_disagreements.csv',index=False)\nprint(out.to_string(index=False))\n")
env = dict(os.environ, OPENBLAS_NUM_THREADS='1')
subprocess.run([sys.executable, str(root/'analyse.py')], env=env, check=True)


In [ ]:
import pandas as pd, shutil
display(pd.read_csv(root/'bootstrap_intervals.csv'))
display(pd.read_csv(root/'per_class_full_data.csv'))
shutil.make_archive('/content/Bootstrap_Results', 'zip', root)
files.download('/content/Bootstrap_Results.zip')